# Esempio di analisi — Divorzio in Italia

Questo notebook mostra come usare i CSV di questo repository per rispondere a tre domande comuni:

1. Come è evoluto il totale dei divorzi in Italia?
2. Quanto dura un procedimento di separazione giudiziale?
3. Chi presenta la domanda di divorzio: il marito o la moglie?

Il notebook usa solo la **libreria standard Python**. Una sezione opzionale in fondo mostra gli stessi esempi con `pandas` per chi preferisce.

> ⚠️ **Attenzione ai filtri**: i CSV includono più livelli di aggregazione (Italia + macroregioni + regioni) e indicatori decomposti (totale + giudiziale + consensuale). Sommare senza filtrare produce stime gonfiate fino a 6x. Vedi [`docs/methodology.md`](../docs/methodology.md).

> Fonte dati: ISTAT, via [open-data-divorzio-italia](https://github.com/divorziare/open-data-divorzio-italia). Licenza CC BY 4.0.

In [ ]:
import csv
from pathlib import Path

REPO = Path('..').resolve()
CSV = REPO / 'data' / 'csv'

def read(path):
    with path.open(encoding='utf-8') as fh:
        return list(csv.DictReader(fh))

## 1. Totale divorzi in Italia, anno per anno

Estrazione della "riga grand total" da `eta-al-divorzio-marito.csv`: Italia + indicatore totale + tutte le altre dimensioni al livello di aggregazione.

In [ ]:
marito = read(CSV / 'eta-al-divorzio-marito.csv')

totali = {}
for r in marito:
    if r['Territorio'] != 'Italia': continue
    if r['Indicatore'] != 'divorzi concessi': continue
    if r['Luogo di nascita del marito'] != 'Totale': continue
    if r['Luogo di nascita della moglie'] != 'Totale': continue
    if r['Luogo di residenza del marito'] != 'Totale': continue
    if r['Luogo di residenza della moglie'] != 'Totale': continue
    if r['Classe di età del marito al divorzio'] != '15 anni e più': continue
    if r['Classe di età della moglie al divorzio'] != '15 anni e più': continue
    if r['Classe di età del marito al matrimonio'] != '15 anni e più': continue
    if r['Classe di età della moglie al matrimonio'] != '15 anni e più': continue
    if r['Anno di matrimonio'] != 'tutte le voci': continue
    if not r['VALUE']: continue
    totali[int(r['TIME_PERIOD'])] = int(r['VALUE'])

for anno in sorted(totali):
    print(f'{anno}: {totali[anno]:>7,} divorzi concessi')

## 2. Durata media del procedimento di separazione giudiziale

Dal file `procedimenti-e-separazioni.csv`, filtro l'indicatore di durata e il territorio nazionale.

In [ ]:
proc = read(CSV / 'procedimenti-e-separazioni.csv')

target = 'durata media del procedimento di separazione giudiziale (in giorni)'
durata = {
    int(r['TIME_PERIOD']): int(r['VALUE'])
    for r in proc
    if r['Territorio'] == 'Italia'
    and r['Indicatore'] == target
    and r['VALUE']
}

for anno in sorted(durata):
    print(f'{anno}: {durata[anno]} giorni')

## 3. Chi presenta la domanda di divorzio giudiziale?

Uso `coniuge-che-ha-presentato-la-domanda-di-divorzio-con-rito-giudiziale.csv`. L'ultimo dataflow è `27_945_DF_DCIS_DIVFIG1_8` (corrente).

In [ ]:
richiedente = read(CSV / 'coniuge-che-ha-presentato-la-domanda-di-divorzio-con-rito-giudiziale.csv')

# Ultimo anno disponibile nella serie corrente
anni = [int(r['TIME_PERIOD']) for r in richiedente
        if r['SOURCE'] == '27_945_DF_DCIS_DIVFIG1_8' and r['VALUE']]
if anni:
    ultimo = max(anni)
    print(f'Ultimo anno disponibile (serie corrente): {ultimo}\n')

# Il campo 'Coniuge che ha presentato la domanda di divorzio con rito giudiziale'
# tipicamente contiene: 'marito', 'moglie', 'congiuntamente', 'tutte le voci'
# Per il breakdown, mantieni Italia + totale + tutte altre dim a 'totale'/'tutte le voci'
# e fai variare solo il richiedente.
col_rich = 'Coniuge che ha presentato la domanda di divorzio con rito giudiziale'
richiedenti = sorted({r[col_rich] for r in richiedente if r[col_rich]})
print('Valori del campo "richiedente":', richiedenti)

## Appendice — stessi esempi con pandas

Se hai `pandas` installato (`pip install pandas`):

In [ ]:
# import pandas as pd
#
# df = pd.read_csv(CSV / 'eta-al-divorzio-marito.csv')
#
# mask = (
#     (df['Territorio'] == 'Italia') &
#     (df['Indicatore'] == 'divorzi concessi') &
#     (df['Luogo di nascita del marito'] == 'Totale') &
#     (df['Luogo di nascita della moglie'] == 'Totale') &
#     (df['Luogo di residenza del marito'] == 'Totale') &
#     (df['Luogo di residenza della moglie'] == 'Totale') &
#     (df['Classe di età del marito al divorzio'] == '15 anni e più') &
#     (df['Classe di età della moglie al divorzio'] == '15 anni e più') &
#     (df['Classe di età del marito al matrimonio'] == '15 anni e più') &
#     (df['Classe di età della moglie al matrimonio'] == '15 anni e più') &
#     (df['Anno di matrimonio'] == 'tutte le voci')
# )
# serie = df[mask].sort_values('TIME_PERIOD').set_index('TIME_PERIOD')['VALUE']
# serie.plot(title='Divorzi concessi in Italia')